In [1]:
%pip install pandas pyarrow requests tqdm python-dateutil

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd 
import requests 
from pathlib  import Path 
from datetime import datetime 
import zipfile
import io
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
diretorio_base = Path(r"E:\2. budget_educacao")
diretorio_download = diretorio_base / "dados_despesas_diarias"
diretorio_download.mkdir(exist_ok=True)

In [ ]:
datas = pd.date_range(start="2021-03-04", end="2021-12-31", freq="D")

import time

link_base = [
    f"https://dadosabertos-download.cgu.gov.br/PortalDaTransparencia/saida/despesas/{d.strftime('%Y%m%d')}_Despesas.zip"
    for d in datas
]

headers = {"User-Agent": "Mozilla/5.0"}
pausa_entre_arquivos = 5        # segundos entre um download e outro
pausa_bloqueio = 5 * 60        # 15 minutos quando o servidor bloquear
max_tentativas = 2

for url in link_base:
    arquivo = diretorio_download / url.split("/")[-1]
    if arquivo.exists():
        print(f"ja tem: {arquivo.name}")
        continue

    baixou = False
    for tentativa in range(1, max_tentativas + 1):
        try:
            print(f"baixando: {arquivo.name} (tentativa {tentativa})")
            r = requests.get(url, headers=headers, timeout=60)
        except Exception as e:
            print(f"Erro: {e}")
            time.sleep(60)
            continue

        if r.status_code == 200:
            with open(arquivo, "wb") as f:
                f.write(r.content)
            baixou = True
            break
        elif r.status_code in (403, 405, 429):
            print(f"bloqueado (HTTP {r.status_code}). esperando {pausa_bloqueio // 60} min...")
            time.sleep(pausa_bloqueio)
        else:
            print(f"erro no http {r.status_code} pulando arquivo")
            break

    if not baixou and r.status_code in (403, 405, 429):
        print("continua bloqueando, parando o script")
        break

    time.sleep(pausa_entre_arquivos)

ja tem: 20210304_Despesas.zip
ja tem: 20210305_Despesas.zip
ja tem: 20210306_Despesas.zip
ja tem: 20210307_Despesas.zip
ja tem: 20210308_Despesas.zip
ja tem: 20210309_Despesas.zip
ja tem: 20210310_Despesas.zip
ja tem: 20210311_Despesas.zip
ja tem: 20210312_Despesas.zip
ja tem: 20210313_Despesas.zip
ja tem: 20210314_Despesas.zip
ja tem: 20210315_Despesas.zip
ja tem: 20210316_Despesas.zip
ja tem: 20210317_Despesas.zip
ja tem: 20210318_Despesas.zip
ja tem: 20210319_Despesas.zip
ja tem: 20210320_Despesas.zip
ja tem: 20210321_Despesas.zip
ja tem: 20210322_Despesas.zip
ja tem: 20210323_Despesas.zip
ja tem: 20210324_Despesas.zip
ja tem: 20210325_Despesas.zip
ja tem: 20210326_Despesas.zip
ja tem: 20210327_Despesas.zip
ja tem: 20210328_Despesas.zip
ja tem: 20210329_Despesas.zip
ja tem: 20210330_Despesas.zip
ja tem: 20210331_Despesas.zip
ja tem: 20210401_Despesas.zip
ja tem: 20210402_Despesas.zip
ja tem: 20210403_Despesas.zip
ja tem: 20210404_Despesas.zip
ja tem: 20210405_Despesas.zip
ja tem: 20

In [ ]:
datas = pd.date_range(start="2021-03-04", end="2021-12-31", freq="D")

link_base = [
    f"https://dadosabertos-download.cgu.gov.br/PortalDaTransparencia/saida/despesas/{d.strftime('%Y%m%d')}_Despesas.zip"
    for d in datas
]

for url in link_base:
    arquivo = diretorio_download / url.split("/")[-1]
    if arquivo.exists():
        print(f"Já existe: {arquivo.name}")
        continue
    try:
        print(f"Baixando: {arquivo.name}")
        r = requests.get(url, timeout=60)
        if r.status_code == 200:
            with open(arquivo, "wb") as f:
                f.write(r.content)
        else:
            print(f"Erro HTTP {r.status_code}")
    except Exception as e:
        print(f"Erro: {e}")


Baixando: 20210304_Despesas.zip
Baixando: 20210305_Despesas.zip
Baixando: 20210306_Despesas.zip
Já existe: 20210307_Despesas.zip
Baixando: 20210308_Despesas.zip
Baixando: 20210309_Despesas.zip
Baixando: 20210310_Despesas.zip
Baixando: 20210311_Despesas.zip
Baixando: 20210312_Despesas.zip
Baixando: 20210313_Despesas.zip
Baixando: 20210314_Despesas.zip
Baixando: 20210315_Despesas.zip
Baixando: 20210316_Despesas.zip
Baixando: 20210317_Despesas.zip
Baixando: 20210318_Despesas.zip
Baixando: 20210319_Despesas.zip
Baixando: 20210320_Despesas.zip
Baixando: 20210321_Despesas.zip
Baixando: 20210322_Despesas.zip
Baixando: 20210323_Despesas.zip
Baixando: 20210324_Despesas.zip
Baixando: 20210325_Despesas.zip
Baixando: 20210326_Despesas.zip
Baixando: 20210327_Despesas.zip
Baixando: 20210328_Despesas.zip
Baixando: 20210329_Despesas.zip
Baixando: 20210330_Despesas.zip
Baixando: 20210331_Despesas.zip
Baixando: 20210401_Despesas.zip
Baixando: 20210402_Despesas.zip
Baixando: 20210403_Despesas.zip
Baixand

KeyboardInterrupt: 

In [16]:
def processar_zip(zip_path: Path, pasta_parquet: Path) -> str:
    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            csvs = [n for n in z.namelist() if n.endswith("_Empenho.csv")]
            for nome_csv in csvs:
                with z.open(nome_csv) as f:
                    df = pd.read_csv(
                        io.TextIOWrapper(f, encoding="latin1"),
                        sep=";",
                        low_memory=False,
                    )
                parquet_name = Path(nome_csv).stem + ".parquet"
                df.to_parquet(pasta_parquet / parquet_name, engine="pyarrow", index=False)
        return f"✓ {zip_path.name}"
    except Exception as e:
        return f"✗ {zip_path.name}: {e}"

pasta_parquet = diretorio_download / "dados_despesas_diarias_parquet"
pasta_parquet.mkdir(parents=True, exist_ok=True)

zips = list(diretorio_download.glob("*.zip"))
print(f"{len(zips)} arquivos encontrados.\n")

with ThreadPoolExecutor(max_workers=4) as executor:
    futuros = {executor.submit(processar_zip, z, pasta_parquet): z for z in zips}
    for i, futuro in enumerate(as_completed(futuros), 1):
        print(f"[{i}/{len(zips)}] {futuro.result()}")

730 arquivos encontrados.

[1/730] ✓ 20230101_Despesas.zip
[2/730] ✓ 20230102_Despesas.zip
[3/730] ✓ 20230103_Despesas.zip
[4/730] ✓ 20230104_Despesas.zip
[5/730] ✓ 20230107_Despesas.zip
[6/730] ✓ 20230105_Despesas.zip
[7/730] ✓ 20230108_Despesas.zip
[8/730] ✓ 20230106_Despesas.zip
[9/730] ✓ 20230109_Despesas.zip
[10/730] ✓ 20230110_Despesas.zip
[11/730] ✓ 20230111_Despesas.zip
[12/730] ✓ 20230114_Despesas.zip
[13/730] ✓ 20230112_Despesas.zip
[14/730] ✓ 20230115_Despesas.zip
[15/730] ✓ 20230113_Despesas.zip
[16/730] ✓ 20230117_Despesas.zip
[17/730] ✓ 20230118_Despesas.zip
[18/730] ✓ 20230116_Despesas.zip
[19/730] ✓ 20230122_Despesas.zip
[20/730] ✓ 20230121_Despesas.zip
[21/730] ✓ 20230119_Despesas.zip
[22/730] ✓ 20230120_Despesas.zip
[23/730] ✓ 20230123_Despesas.zip
[24/730] ✓ 20230125_Despesas.zip
[25/730] ✓ 20230124_Despesas.zip
[26/730] ✓ 20230128_Despesas.zip
[27/730] ✓ 20230129_Despesas.zip
[28/730] ✓ 20230126_Despesas.zip
[29/730] ✓ 20230127_Despesas.zip
[30/730] ✓ 20230130_Despe

In [17]:
pasta_anual = pasta_parquet / "anual"
pasta_anual.mkdir(exist_ok=True)

anos = ["2023", "2024"]

for ano in anos:

    arquivos_ano = sorted(
        pasta_parquet.glob(f"{ano}*_Despesas_Empenho.parquet")
    )

    print(f"\nAno {ano}")
    print(f"{len(arquivos_ano)} arquivos")

    dfs = []

    for arq in arquivos_ano:

        try:
            df = pd.read_parquet(arq)

            # equivalente ao as.character()
            df = df.astype(str)

            dfs.append(df)

        except Exception as e:
            print(arq.name, e)

    if len(dfs):

        dados_ano = pd.concat(
            dfs,
            ignore_index=True
        )

        destino = pasta_anual / f"despesas_empenho_{ano}.parquet"

        dados_ano.to_parquet(
            destino,
            index=False
        )

        print("salvo:", destino.name)


Ano 2023
364 arquivos
salvo: despesas_empenho_2023.parquet

Ano 2024
366 arquivos
salvo: despesas_empenho_2024.parquet


In [18]:
list(pasta_anual.glob("*.parquet"))

df_teste = pd.read_parquet(
    pasta_anual / "despesas_empenho_2023.parquet"
)

print(df_teste.shape)

print(df_teste.columns.tolist())

df_teste.head()

(1783391, 63)
['Id Empenho', 'Código Empenho', 'Código Empenho Resumido', 'Data Emissão', 'Código Tipo Documento', 'Tipo Documento', 'Tipo Empenho', 'Espécie Empenho', 'Código Órgão Superior', 'Órgão Superior', 'Código Órgão', 'Órgão', 'Código Unidade Gestora', 'Unidade Gestora', 'Código Gestão', 'Gestão', 'Código Favorecido', 'Favorecido', 'Observação', 'Código Esfera Orçamentária', 'Esfera Orçamentária', 'Código Tipo Crédito', 'Tipo Crédito', 'Código Grupo Fonte Recurso', 'Grupo Fonte Recurso', 'Código Fonte Recurso', 'Fonte Recurso', 'Código Unidade Orçamentária', 'Unidade Orçamentária', 'Código Função', 'Função', 'Código SubFunção', 'SubFunção', 'Código Programa', 'Programa', 'Código Ação', 'Ação', 'Linguagem Cidadã', 'Código Subtítulo (Localizador)', 'Subtítulo (Localizador)', 'Código Plano Orçamentário', 'Plano Orçamentário', 'Código Programa Governo', 'Nome Programa Governo', 'Autor Emenda', 'Código Categoria de Despesa', 'Categoria de Despesa', 'Código Grupo de Despesa', 'Grupo

,Id Empenho,Código Empenho,Código Empenho Resumido,Data Emissão,Código Tipo Documento,Tipo Documento,Tipo Empenho,Espécie Empenho,Código Órgão Superior,Órgão Superior,...,Processo,Modalidade de Licitação,Inciso,Amparo,Referência de Dispensa ou Inexigibilidade,Código Convênio,Contrato de Repasse / Termo de Parceria / Outros,Valor Original do Empenho,Valor do Empenho Convertido pra R$,Valor Utilizado na Conversão
0,515902249,167233000012023NE000270,2023NE000270,01/01/2023,NE,Nota de Empenho,Global,Não se aplica,52000,Ministério da Defesa,...,64294.001707/2023-62,Dispensa de Licitação,II,LEI 8.666 / 1993,-2,-1,NAO SE APLICA,"4680,00","4680,00","1,0000"
1,526501194,250025000012022NE000123,2022NE000123,01/01/2023,NE,Nota de Empenho,Estimativo,Não se aplica,36000,Ministério da Saúde,...,25003.001658/2022-53,Pregão,SI,LEI 10.520 / 2002,-2,-1,NAO SE APLICA,"0,00","0,00","0,00"
2,561772761,167118000012023NE001715,2023NE001715,01/01/2023,NE,Nota de Empenho,Estimativo,Não se aplica,52000,Ministério da Defesa,...,64316.128466/2023-01,Inexigível,SI,LEI 8.666 / 1993,-2,-1,NAO SE APLICA,"3800,00","3800,00","1,0000"
3,515001629,152005000012023NE000045,2023NE000045,01/01/2023,NE,Nota de Empenho,Global,Não se aplica,26000,Ministério da Educação,...,23121.001232/2018-41,Pregão,SI,LEI 10.520 / 2002,-2,-1,NAO SE APLICA,"34944,00","34944,00","1,0000"
4,520401565,366003362102023NE000080,2023NE000080,02/01/2023,NE,Nota de Empenho,Ordinário,Não se aplica,36000,Ministério da Saúde,...,1328/22,Dispensa de Licitação,XV,LEI 13.303 / 2016,-2,-1,NAO SE APLICA,"1495,00","1495,00","1,0000"


In [22]:
dados_2023 = pd.read_parquet(
    pasta_anual / "despesas_empenho_2023.parquet"
)

dados_2024 = pd.read_parquet(
    pasta_anual / "despesas_empenho_2024.parquet"
)

print(dados_2023.shape)
print(dados_2024.shape)

dados_2023["Código SubFunção"] = pd.to_numeric(
    dados_2023["Código SubFunção"],
    errors="coerce"
)

dados_2024["Código SubFunção"] = pd.to_numeric(
    dados_2024["Código SubFunção"],
    errors="coerce"
)

vigilancia_2023 = dados_2023[
    dados_2023["Código SubFunção"] == 305
]

vigilancia_2024 = dados_2024[
    dados_2024["Código SubFunção"] == 305
]

print(vigilancia_2023.shape)
print(vigilancia_2024.shape)

dados_2023 = pd.read_parquet(
    pasta_anual / "despesas_empenho_2023.parquet"
)

vigilancia_2023 = dados_2023[
    pd.to_numeric(
        dados_2023["Código SubFunção"],
        errors="coerce"
    ) == 305
]

del dados_2023


dados_2024 = pd.read_parquet(
    pasta_anual / "despesas_empenho_2024.parquet"
)

vigilancia_2024 = dados_2024[
    pd.to_numeric(
        dados_2024["Código SubFunção"],
        errors="coerce"
    ) == 305
]

del dados_2024

print(vigilancia_2023.shape)
print(vigilancia_2024.shape)


(1783391, 63)
(1764026, 63)
(6491, 63)
(7409, 63)
(6491, 63)
(7409, 63)


In [23]:
dados_vigilancia = pd.concat(
    [
        vigilancia_2023,
        vigilancia_2024
    ],
    ignore_index=True
)

print(dados_vigilancia.shape)

(13900, 63)


In [24]:
dados_vigilancia["Data Emissão"] = pd.to_datetime(
    dados_vigilancia["Data Emissão"],
    dayfirst=True,
    errors="coerce"
)

dados_vigilancia["ano"] = (
    dados_vigilancia["Data Emissão"]
    .dt.year
)

dados_vigilancia["mes"] = (
    dados_vigilancia["Data Emissão"]
    .dt.month
)

dados_vigilancia["semana"] = (
    dados_vigilancia["Data Emissão"]
    .dt.isocalendar()
    .week
)

In [26]:
dados_vigilancia["Espécie Empenho"].value_counts(dropna=False)


dados_vigilancia["Valor Original do Empenho"].value_counts(dropna=False)

Valor Original do Empenho
0,00         925
100000,00     98
150000,00     80
110000,00     78
165000,00     73
            ... 
69694,00       1
34992,00       1
20332,69       1
1036,00        1
34986,00       1
Name: count, Length: 8072, dtype: int64

In [27]:
dados_vigilancia.head()

,Id Empenho,Código Empenho,Código Empenho Resumido,Data Emissão,Código Tipo Documento,Tipo Documento,Tipo Empenho,Espécie Empenho,Código Órgão Superior,Órgão Superior,...,Amparo,Referência de Dispensa ou Inexigibilidade,Código Convênio,Contrato de Repasse / Termo de Parceria / Outros,Valor Original do Empenho,Valor do Empenho Convertido pra R$,Valor Utilizado na Conversão,ano,mes,semana
0,528601534,250005000012023NE000009,2023NE000009,2023-01-05,NE,Nota de Empenho,Global,Não se aplica,36000,Ministério da Saúde,...,LEI 10.520 / 2002,-2,-1,NAO SE APLICA,"33132,24","33132,24","1,0000",2023,1,1
1,509201583,250005000012023NE000010,2023NE000010,2023-01-05,NE,Nota de Empenho,Global,Não se aplica,36000,Ministério da Saúde,...,LEI 10.520 / 2002,-2,-1,NAO SE APLICA,"32916,98","32916,98","1,0000",2023,1,1
2,509401461,257001000012023NE440242,2023NE440242,2023-01-05,NE,Nota de Empenho,Global,Não se aplica,36000,Ministério da Saúde,...,SEM INFORMACAO,-2,-1,NAO SE APLICA,"35623,65","35623,65","1,0000",2023,1,1
3,523101510,257001000012023NE440243,2023NE440243,2023-01-05,NE,Nota de Empenho,Global,Não se aplica,36000,Ministério da Saúde,...,SEM INFORMACAO,-2,-1,NAO SE APLICA,"297709,58","297709,58","1,0000",2023,1,1
4,519901571,257001000012023NE440238,2023NE440238,2023-01-05,NE,Nota de Empenho,Global,Não se aplica,36000,Ministério da Saúde,...,SEM INFORMACAO,-2,-1,NAO SE APLICA,"35000,00","35000,00","1,0000",2023,1,1


In [28]:
dados_vigilancia_filtro = dados_vigilancia[
    (
        dados_vigilancia["Data Emissão"] >= "2023-10-02"
    )
    &
    (
        dados_vigilancia["Data Emissão"] <= "2024-09-27"
    )
].copy()

print(dados_vigilancia_filtro.shape)

(6743, 66)


In [29]:
dados_vigilancia_filtro["Valor Original do Empenho"] = (
    pd.to_numeric(
        dados_vigilancia_filtro["Valor Original do Empenho"]
        .astype(str)
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )
)

dados_vigilancia_filtro["Valor do Empenho Convertido pra R$"] = (
    pd.to_numeric(
        dados_vigilancia_filtro["Valor do Empenho Convertido pra R$"]
        .astype(str)
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )
)

In [30]:
dados_vigilancia_filtro[
    [
        "Data Emissão",
        "Valor Original do Empenho",
        "Valor do Empenho Convertido pra R$"
    ]
].head()

,Data Emissão,Valor Original do Empenho,Valor do Empenho Convertido pra R$
4275,2023-10-02,12540.00,12540.00
4276,2023-10-02,48133.05,48133.05
4277,2023-10-02,8550.00,8550.00
4278,2023-10-02,79306.16,79306.16
4279,2023-10-02,15829.58,15829.58


In [31]:
dados_vigilancia_filtro.to_csv(
    diretorio_base / "banco_diario_artigo_gustavo.csv",
    index=False,
    encoding="utf-8-sig"
)